# Deriving Upper Bound on Cost for Hypervolume Metric (ConstantPricesScenario)

This notebook gives a walkthrough to the computation of the **largest conceivable daily cost** for the ConstantPricesScenario **with (Case 1) and without (Case 2) a heat pump**. We will use the CSV data sources from the `./commonpower/finetuning/data/` directory provided in the project:

* `ICLR_load.csv` – baseline building load (MW)
* `ICLR_pv.csv` – PV generation (MW)
* `ToU_prices.csv` – grid buying price (€/MWh)

The result of this notebook will serve as the **lower bound on the reward function** (because reward = −cost) for the PCN approach when computing the hypervolume metric.

### Imports / File Paths / CSV Parsing

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path().resolve().parent / 'finetuning/data'

LOAD_FILE = BASE_DIR / 'ICLR_load.csv'
PV_FILE = BASE_DIR / 'ICLR_pv.csv'
PRICE_FILE = BASE_DIR / 'ToU_prices.csv'

#####################################################################

load = pd.read_csv(LOAD_FILE, parse_dates=['t']).rename(columns={'p':'load_MW'})
pv   = pd.read_csv(PV_FILE,   parse_dates=['t']).rename(columns={'p':'pv_MW'})
prices = (pd.read_csv(PRICE_FILE, parse_dates=['t'])
            .rename(columns={'psib_mwh':'price_EUR_per_MWh'}))

for df, name in [(load,'Load'), (pv,'PV'), (prices,'Prices')]:
    print(f"{name:7}: {df.shape[0]:5d} rows, time span {df['t'].min()} → {df['t'].max()}")

Load   :  8784 rows, time span 2016-01-01 00:00:00 → 2016-12-31 23:00:00
PV     :  8784 rows, time span 2016-01-01 00:00:00 → 2016-12-31 23:00:00
Prices :  8784 rows, time span 2016-01-01 00:00:00 → 2016-12-31 23:00:00


> **FYI - Unit conventions**
>
> * Load & PV columns are given in **MW** (Megawatts).  
> * Prices are in **€/MWh**. To convert to **€/kWh** divide by 1000.  
> * Each row represents a **1‑hour** interval. Therefore the energy purchased in that interval is equal to `Net_Load_kW * 1 h = kWh`.

# Case 1: Lower Bound without a Heat Pump

We compute the worst-case daily electricity cost based on building base load, PV generation, and time-of-use grid prices (`ToU_prices.csv`).
The largest daily cost (upper-bound cost ⇒ lower-bound reward) in 2016 is thus calculated.

### 1. Net Load (MW)
For each 1h interval $t$ we determine the power that must be taken from or exported to the grid by calculating:

$\text{Net Load}_{t}\;[\text{MW}] \;=\; \text{Load}_{t}\;[\text{MW}] \;-\; \text{PV}_{t}\;[\text{MW}]$

### 2. Interval Cost (Euro)

We want an upper-bound (thus worst-case) cost so we:

* **pay** for every positive net-load (imports)
* **do not earn** revenue for exports (negative net-load is zero-clipped)

With hourly data the conversion is straightforward:

$$
\text{Cost}_{t} \;=\; 
\max\!\bigl(0,\text{Net Load}_{t}\bigr)
\;\cdot\;
\text{Price}_{t} \;[\text{Euro}/ \text{MWh}]
$$

*Note that because $MW \cdot 1 h = MWh$, we can directly use that formula without extra scaling what so ever.*

### 3. Daily Cost (Euro)

Sum all 24 hourly costs in a day:

$$
\text{Daily Cost}_{d} 
\;=\;
\sum_{t\in d}\text{Cost}_{t}
$$

### 4. Maximum Daily Cost

Find the largest $\text{Daily Cost}_{d}$ over the whole year.

## 1. Compute Net Load and Preprocessing 

In [ ]:
df = (load.merge(pv, on='t', how='inner').merge(prices[['t','price_EUR_per_MWh']], on='t'))

# Convert MW -> kW so that multiplying by hours = kWh
df['load_kW'] = df['load_MW'] * 1_000
df['pv_kW']   = df['pv_MW']   * 1_000

# Net load (positive means we import from the grid, negative means we export)
df['net_load_kW'] = df['load_kW'] - df['pv_kW']

## 2. Worst‑Case Interval Cost
For the *upper‑bound* (worst case) we assume:

* We pay for every positive net kW (imports).
* We do not receive credits for exporting (-> negative net load is capped at 0).

Interval cost in Euro:
` Cost_interval = max(0, net_load_kW) * (price_EUR_per_MWh / 1000) `

In [3]:
df['price_EUR_per_kWh'] = df['price_EUR_per_MWh'] / 1_000
df['interval_cost_EUR'] = df['net_load_kW'].clip(lower=0) * df['price_EUR_per_kWh']

print("Interval cost statistics (EUR):")
print(df['interval_cost_EUR'].describe(percentiles=[0.9, 0.99]))

Interval cost statistics (EUR):
count    8784.000000
mean        2.849758
std         4.172849
min        -4.637646
50%         1.195215
90%         8.157311
99%        18.322361
max        44.494380
Name: interval_cost_EUR, dtype: float64


## 3. Aggregate to Daily Cost

In [4]:
df['date'] = df['t'].dt.date
daily_cost = df.groupby('date')['interval_cost_EUR'].sum().sort_values(ascending=False)

print(f"Max daily cost: {daily_cost.iloc[0]:.2f} € on {daily_cost.index[0]}")

daily_cost.head()

Max daily cost: 324.57 € on 2016-12-06


date
2016-12-06    324.569755
2016-12-04    323.986229
2016-12-16    307.204150
2016-01-21    302.518262
2016-12-14    279.989112
Name: interval_cost_EUR, dtype: float64

Therefore the **Result for the lower bound** in Case 1 is given by: **€ 324.57** on **2016-12-06**

# Case 2: Lower Bound with a Heat Pump

Now we are adding the electrical consumption of the described heat pump. The result gives a **lower bound on reward** (upper bound on cost + penalty) again for hyper‑volume metrics. Again we need the Data sources as above:

* `ICLR_load.csv` – baseline building load (MW)
* `ICLR_pv.csv` – PV generation (MW)
* `ToU_prices.csv` – grid buying price (€/MWh)

For the heat pump case we also need to consider:
* `DE_Temperature_and_COP2016.csv` – outside temperature and heat‑pump COP (hourly)


### 1. Simplified Heat-Pump Electrical Power Cost

To simplify the calculation, we use a fixed maximum power consumption for the heat pump and multiply it by the cost of electricity over the maximum time period.

$$ \text{Heat Pump Cost} = P_{\text{hp,max}} \cdot \text{Price} \cdot \text{Time} $$

Where:
* $P_{\text{hp,max}}$ = 5 kW (the maximum power of the heat pump as defined in `scenarios.py`)
* Price is the electricity price in €/kWh
* Time is the duration in hours


In [ ]:
P_HP_MAX = 5.0  # kW
EPISODE_H = 24  # hours

# Calculate additional cost from heat pump
df['heat_pump_cost_EUR'] = P_HP_MAX * df['price_EUR_per_kWh'] * EPISODE_H

# Add hp cost to daily cost
daily_cost_with_hp = daily_cost + df.groupby(df['t'].dt.date)['heat_pump_cost_EUR'].sum()

# Find worst case daily cost
worst_cost_with_hp = daily_cost_with_hp.max()
worst_day_with_hp = daily_cost_with_hp.idxmax()

print(f"Max daily cost with heat pump: €{worst_cost_with_hp:,.2f} on {worst_day_with_hp}")

Max daily cost with heat pump: €474.45 on 2016-12-06
